<a href="https://colab.research.google.com/github/dzidz1/Freeuni_ML_Facial_Expression_Recognition/blob/main/03_resnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup, data and training utilities

In [ ]:
!pip install -q --upgrade kaggle wandb
import os, math, copy, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from google.colab import userdata
import wandb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
wandb.login(key=userdata.get('WANDB_API_KEY'))
print("Device:", device)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Device: cuda


In [ ]:
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -q -o challenges-in-representation-learning-facial-expression-recognition-challenge.zip

challenges-in-representation-learning-facial-expression-recognition-challenge.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
df = pd.read_csv('icml_face_data.csv')
df.columns = df.columns.str.strip()
df['Usage'] = df['Usage'].str.strip()

def pixels_to_array(s):
    return np.array([np.array(p.split(), dtype=np.uint8) for p in s]).reshape(-1, 48, 48)

class FERDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images, self.labels, self.transform = images, labels, transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img = torch.from_numpy(self.images[idx]).float().unsqueeze(0) / 255.0
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])

splits = {}
for name, usage in [('train','Training'), ('val','PublicTest'), ('test','PrivateTest')]:
    sub = df[df['Usage'] == usage]
    splits[name] = FERDataset(pixels_to_array(sub['pixels']), sub['emotion'].values)

BATCH = 64
train_loader = DataLoader(splits['train'], batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(splits['val'],   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(splits['test'],  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print("Train/Val/Test:", len(splits['train']), len(splits['val']), len(splits['test']))

Train/Val/Test: 28709 3589 3589


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train(); rl, c, t = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x); loss = criterion(out, y)
        loss.backward(); optimizer.step()
        rl += loss.item()*x.size(0); c += (out.argmax(1)==y).sum().item(); t += y.size(0)
    return rl/t, c/t

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval(); rl, c, t = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x); loss = criterion(out, y)
        rl += loss.item()*x.size(0); c += (out.argmax(1)==y).sum().item(); t += y.size(0)
    return rl/t, c/t

def run_experiment(model, run_name, group, config, train_loader, val_loader, device, patience=None, use_scheduler=False):
    wandb.init(project="fer2013-emotion-recognition", name=run_name, group=group, config=config)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config.get("weight_decay", 0))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3) if use_scheduler else None
    best_val_loss, best_val_acc, best_state, best_epoch, wait = float('inf'), 0.0, None, 0, 0
    for epoch in range(1, config["epochs"]+1):
        tl, ta = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl, va = evaluate(model, val_loader, criterion, device)
        if scheduler:
            scheduler.step(vl)
        lr_now = optimizer.param_groups[0]['lr']
        wandb.log({"epoch": epoch, "train_loss": tl, "train_acc": ta, "val_loss": vl, "val_acc": va, "lr": lr_now})
        print(f"Epoch {epoch:2d}/{config['epochs']} | train {tl:.4f}/{ta:.4f} | val {vl:.4f}/{va:.4f} | lr {lr_now:.1e}")
        best_val_acc = max(best_val_acc, va)
        if vl < best_val_loss:
            best_val_loss, best_state, best_epoch, wait = vl, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            wait += 1
            if patience and wait >= patience:
                print(f"Early stopping at epoch {epoch} (best val_loss was epoch {best_epoch})")
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    wandb.run.summary.update({"best_val_acc": best_val_acc, "best_val_loss": best_val_loss, "best_epoch": best_epoch})
    wandb.finish()
    return model

## Architecture - ResNet with residual connections

In [ ]:
class ResidualBlock(nn.Module):
  def __init__(self, in_ch, out_ch, stride=1):
    super().__init__()
    self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
    self.bn1 = nn.BatchNorm2d(out_ch)
    self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
    self.bn2 = nn.BatchNorm2d(out_ch)
    self.shortcut = nn.Sequential()
    if stride != 1 or in_ch != out_ch:
      self.shortcut = nn.Sequential(
          nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
          nn.BatchNorm2d(out_ch)
      )

  def forward(self, x):
    out = F.relu(self.bn1(self.conv1(x)))
    out = self.bn2(self.conv2(out))
    out += self.shortcut(x)
    return F.relu(out)

class ResNetMini(nn.Module):
  def __init__(self, num_classes=7, dropout=0.3):
    super().__init__()
    self.stem = nn.Sequential(nn.Conv2d(1, 64, 3, padding=1, bias=False),
                              nn.BatchNorm2d(64), nn.ReLU())
    self.layer1 = ResidualBlock(64, 64)
    self.layer2 = ResidualBlock(64, 128, stride=2)
    self.layer3 = ResidualBlock(128, 256, stride=2)
    self.pool = nn.AdaptiveAvgPool2d(1)
    self.dropout = nn.Dropout(dropout)
    self.fc = nn.Linear(256, num_classes)

  def forward(self, x):
    x = self.stem(x)
    x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
    x = self.pool(x).flatten(1)
    return self.fc(self.dropout(x))

print(ResNetMini())

ResNetMini(
  (stem): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (layer1): ResidualBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (shortcut): Sequential()
  )
  (layer2): ResidualBlock(
    (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(128, eps=1e-05, mome

In [ ]:
config = {"architecture": "ResNetMini", "lr": 1e-3, "batch_size": 64, "optimizer": "Adam",
          "weight_decay": 1e-4, "dropout": 0.3, "epochs": 40}
model = ResNetMini(dropout=config["dropout"])
run_experiment(model, run_name="ResNet-regularized", group="ResNet",
               config=config, train_loader=train_loader, val_loader=val_loader, device=device, patience=8)

Epoch  1/40 | train 1.7258/0.2997 | val 1.8189/0.2653
Epoch  2/40 | train 1.4732/0.4336 | val 1.4685/0.4372
Epoch  3/40 | train 1.3202/0.4958 | val 1.9897/0.2736
Epoch  4/40 | train 1.2240/0.5359 | val 1.4778/0.4656
Epoch  5/40 | train 1.1519/0.5624 | val 1.2049/0.5358
Epoch  6/40 | train 1.0891/0.5882 | val 1.2662/0.5255
Epoch  7/40 | train 1.0284/0.6152 | val 1.3221/0.5300
Epoch  8/40 | train 0.9592/0.6434 | val 1.1748/0.5706
Epoch  9/40 | train 0.8940/0.6672 | val 1.2752/0.5419
Epoch 10/40 | train 0.7999/0.7058 | val 1.2208/0.5723
Epoch 11/40 | train 0.6988/0.7455 | val 1.6197/0.5210
Epoch 12/40 | train 0.5871/0.7872 | val 2.1463/0.4249
Epoch 13/40 | train 0.4821/0.8275 | val 1.8060/0.5174
Epoch 14/40 | train 0.3718/0.8679 | val 1.9217/0.5316
Epoch 15/40 | train 0.3028/0.8947 | val 1.7230/0.5598
Epoch 16/40 | train 0.2464/0.9154 | val 2.2486/0.5091
Early stopping at epoch 16 (best val_loss was epoch 8)


epoch,▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
train_acc,▁▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_loss,█▇▆▆▅▅▅▄▄▄▃▃▂▂▁▁
val_acc,▁▅▁▆▇▇▇█▇█▇▅▇▇█▇
val_loss,▅▃▆▃▁▂▂▁▂▁▄▇▅▆▅█
best_epoch,8
best_val_acc,0.5723
best_val_loss,1.17479
epoch,16
train_acc,0.91536
train_loss,0.24644


ResNetMini(
  (stem): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (layer1): ResidualBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (shortcut): Sequential()
  )
  (layer2): ResidualBlock(
    (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(128, eps=1e-05, mome

In [ ]:
from torchvision import transforms

train_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(12),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
])
aug_train_ds = FERDataset(splits['train'].images, splits['train'].labels, transform=train_aug)
aug_train_loader = DataLoader(aug_train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)

In [ ]:
config = {"architecture": "ResNetMini", "lr": 1e-3, "batch_size": 64, "optimizer": "Adam",
          "weight_decay": 1e-4, "dropout": 0.3, "epochs": 60, "augmentation": True}
model = ResNetMini(dropout=config["dropout"])
run_experiment(model, run_name="ResNet-augmented", group="ResNet",
               config=config, train_loader=aug_train_loader, val_loader=val_loader, device=device, patience=12)

Epoch  1/60 | train 1.7404/0.2866 | val 1.8638/0.1934
Epoch  2/60 | train 1.5478/0.3930 | val 1.4876/0.4249
Epoch  3/60 | train 1.4238/0.4529 | val 1.6334/0.4107
Epoch  4/60 | train 1.3362/0.4891 | val 1.5311/0.4586
Epoch  5/60 | train 1.2713/0.5177 | val 1.2755/0.5088
Epoch  6/60 | train 1.2263/0.5376 | val 1.9146/0.3302
Epoch  7/60 | train 1.1938/0.5482 | val 1.1848/0.5556
Epoch  8/60 | train 1.1595/0.5595 | val 1.3731/0.4873
Epoch  9/60 | train 1.1438/0.5682 | val 1.2897/0.5244
Epoch 10/60 | train 1.1170/0.5771 | val 1.1578/0.5662
Epoch 11/60 | train 1.0979/0.5890 | val 1.2446/0.5366
Epoch 12/60 | train 1.0797/0.5905 | val 1.1278/0.5751
Epoch 13/60 | train 1.0658/0.6008 | val 1.2730/0.5135
Epoch 14/60 | train 1.0604/0.6003 | val 1.2707/0.5389
Epoch 15/60 | train 1.0406/0.6094 | val 1.1305/0.5882
Epoch 16/60 | train 1.0267/0.6118 | val 1.1139/0.5795
Epoch 17/60 | train 1.0177/0.6211 | val 1.1525/0.5676
Epoch 18/60 | train 0.9992/0.6279 | val 1.1960/0.5692
Epoch 19/60 | train 0.9966/0

epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
train_acc,▁▃▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████████
train_loss,█▆▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val_acc,▁▅▄▅▆▃▇▆▆▇▆▇▆▇▇▇▇▇▅███▇▇█▇▇██████
val_loss,█▅▆▅▃█▂▄▃▂▃▂▃▃▂▂▂▂▅▂▁▁▂▂▁▂▂▁▁▁▁▂▁
best_epoch,21
best_val_acc,0.63305
best_val_loss,1.01103
epoch,33
train_acc,0.66969
train_loss,0.88604


ResNetMini(
  (stem): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (layer1): ResidualBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (shortcut): Sequential()
  )
  (layer2): ResidualBlock(
    (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(128, eps=1e-05, mome

In [ ]:
config = {"architecture": "ResNetMini", "lr": 1e-3, "batch_size": 64, "optimizer": "Adam",
          "weight_decay": 1e-4, "dropout": 0.3, "epochs": 60, "augmentation": True, "lr_scheduler": "ReduceLROnPlateau"}
model = ResNetMini(dropout=config["dropout"])
run_experiment(model, run_name="ResNet-augmented-scheduler", group="ResNet",
               config=config, train_loader=aug_train_loader, val_loader=val_loader, device=device,
               patience=12, use_scheduler=True)

Epoch  1/60 | train 1.7275/0.2945 | val 1.7537/0.3282 | lr 1.0e-03
Epoch  2/60 | train 1.5321/0.4020 | val 1.5900/0.3814 | lr 1.0e-03
Epoch  3/60 | train 1.4187/0.4546 | val 1.7658/0.3572 | lr 1.0e-03
Epoch  4/60 | train 1.3423/0.4887 | val 1.3959/0.4628 | lr 1.0e-03
Epoch  5/60 | train 1.2843/0.5098 | val 1.3254/0.4979 | lr 1.0e-03
Epoch  6/60 | train 1.2369/0.5290 | val 1.3731/0.4979 | lr 1.0e-03
Epoch  7/60 | train 1.1958/0.5501 | val 1.2060/0.5447 | lr 1.0e-03
Epoch  8/60 | train 1.1707/0.5566 | val 1.2206/0.5442 | lr 1.0e-03
Epoch  9/60 | train 1.1427/0.5709 | val 1.3149/0.5378 | lr 1.0e-03
Epoch 10/60 | train 1.1259/0.5755 | val 1.1849/0.5478 | lr 1.0e-03
Epoch 11/60 | train 1.1113/0.5796 | val 1.1769/0.5723 | lr 1.0e-03
Epoch 12/60 | train 1.0897/0.5903 | val 1.2335/0.5467 | lr 1.0e-03
Epoch 13/60 | train 1.0743/0.5943 | val 1.0841/0.5893 | lr 1.0e-03
Epoch 14/60 | train 1.0600/0.6021 | val 1.1202/0.5815 | lr 1.0e-03
Epoch 15/60 | train 1.0482/0.6057 | val 1.1165/0.5784 | lr 1.0

epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
lr,████████████████▄▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▁▁▁▁
train_acc,▁▃▃▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇███████
train_loss,█▇▆▅▅▅▅▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
val_acc,▁▂▂▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█████████████
val_loss,█▆█▅▄▅▃▃▄▃▃▃▂▂▂▃▂▁▂▁▂▂▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁
best_epoch,30
best_val_acc,0.66732
best_val_loss,0.95673
epoch,42
lr,3e-05


ResNetMini(
  (stem): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (layer1): ResidualBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (shortcut): Sequential()
  )
  (layer2): ResidualBlock(
    (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(128, eps=1e-05, mome